# 🎙️ LVC Voice 6 — Tạo giọng nói (khách)

Chạy server voice của bạn trên GPU miễn phí của Colab. Chỉ cần **key bản quyền**.

**Cách dùng:** điền key vào ô 1 → menu **Runtime → Run all** → đợi ô 3 in ra
**API URL** và **API key**. Dùng cặp đó gọi tạo giọng từ máy bạn (xem ô cuối).

> **Bản chất lượng cao:** server mặc định **32 bước · engine TỰ chia đoạn**
> (liền mạch, không cắt vụn), nhận tới **8000 ký tự/request**. Cứ gửi cả đoạn —
> để engine tự chia cho giọng mượt. VRAM có trần nhờ guard nên gửi bài dài không
> nổ. Muốn nhanh hơn (giọng gần như không đổi) thì gửi `steps: 16`.

> Để **ô 3 chạy suốt** trong lúc dùng — tắt ô là tắt server. Mất mạng thì chạy
> lại ô 3, lấy URL mới.


In [ ]:
#@title 1. Nhập key bản quyền { display-mode: "form" }
#@markdown Dán key bản quyền của bạn rồi chạy ô này.
LICENSE_KEY = ""  #@param {type:"string"}

import hashlib
LICENSE_KEY = LICENSE_KEY.strip()
assert LICENSE_KEY and not LICENSE_KEY.startswith("XXXX"), "Chưa nhập key bản quyền."

# device_id GẮN VỚI KEY, không theo phần cứng: mỗi máy Colab là một VM mới nên
# fingerprint phần cứng đổi mỗi phiên -> mỗi lần chạy ăn một slot thiết bị của
# license, vài lần là hết. Hash của key thì cố định -> mỗi key luôn là "một
# thiết bị", không churn slot.
DEVICE_ID = "colab-" + hashlib.sha256(LICENSE_KEY.encode()).hexdigest()[:24]
# MẶC ĐỊNH 1 LUỒNG: chạy tuần tự thì pool VRAM chỉ nở đúng một buffer, có trần
# cố định, KHÔNG cộng dồn qua các job -> không OOM dù nhiều câu dài. Nhiều luồng
# nhanh hơn nhưng pool nở theo số luồng song song và không co lại -> dễ cạn VRAM
# rồi crash. Đổi số này lên chỉ khi đã có guard giới hạn VRAM ở server.
WORKERS = 1  #@param {type:"integer"}
print("Key nhận rồi. device_id =", DEVICE_ID, "| WORKERS =", WORKERS)


AssertionError: Chưa nhập key bản quyền.

In [ ]:
# ── 2. Cài thư viện + tải loader bảo mật ──────────────────────────────
import hashlib, urllib.request, importlib.util, sys, subprocess, os, glob

# Cài thư viện mạng + crypto + runtime CUDA 12 cho backend GPU
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "httpx", "cryptography",
                "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"], check=True)

# Nạp tất cả đường dẫn CUDA (cả hệ thống lẫn gói wheel nvidia) vào LD_LIBRARY_PATH
cuda_dirs = (
    glob.glob("/usr/local/cuda*/lib64") +
    glob.glob("/usr/local/cuda*/targets/x86_64-linux/lib") +
    glob.glob(f"{sys.prefix}/lib/python*/site-packages/nvidia/*/lib") +
    glob.glob(f"{os.path.expanduser('~')}/.local/lib/python*/site-packages/nvidia/*/lib") +
    ["/usr/lib/x86_64-linux-gnu"]
)
valid_dirs = [d for d in dict.fromkeys(cuda_dirs) if os.path.isdir(d)]
if valid_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(valid_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

BASE = "https://github.com/GrayPham/submodulevoice.git".replace(".git", "")
SO_URL  = f"{BASE}/releases/download/loader-linux/voice_loader-cp313-linux-x86_64.so"
SHA_URL = SO_URL + ".sha256"
SO_PATH = "/content/voice_loader-cp313-linux-x86_64.so"

print("Tải loader:", SO_URL)
urllib.request.urlretrieve(SO_URL, SO_PATH)
try:
    want = urllib.request.urlopen(SHA_URL, timeout=20).read().decode().split()[0]
    got = hashlib.sha256(open(SO_PATH, "rb").read()).hexdigest()
    assert got == want, f"checksum loader KHÔNG khớp:\n  cần {want}\n  được {got}"
    print("checksum loader khớp.")
except Exception as e:
    print("(bỏ qua kiểm checksum:", e, ")")

# Nạp loader .so vào như module 'voice_loader'.
spec = importlib.util.spec_from_file_location("voice_loader", SO_PATH)
voice_loader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(voice_loader)
print("Loader nạp OK — phiên bản binary, kill-switch bên trong.")


In [ ]:
# ── 3. Đăng nhập + bật server (ĐỂ Ô NÀY CHẠY LIÊN TỤC) ────────────────
# Loader tự: xác thực key -> tải gói mã hoá từ R2 -> giải mã trong RAM ->
# bật server voice -> mở đường hầm -> in API URL + key. Định kỳ gọi về; nếu
# key bị thu hồi thì tự tắt.
rc = voice_loader.run(LICENSE_KEY, DEVICE_ID, WORKERS)
print("Kết thúc, rc =", rc)


## Gọi tạo giọng từ máy bạn (chất lượng cao)

Sau khi ô 3 in ra **API URL** + **API key**, chạy đoạn dưới **trên máy bạn**
(không phải trong Colab — ô 3 đang giữ server). Mặc định **32 bước, engine tự
chia đoạn** nên chỉ cần gửi cả đoạn, KHÔNG tự cắt 250/500 nữa:

```python
import requests, base64, io, soundfile as sf

URL = "https://....trycloudflare.com"   # dán từ ô 3
KEY = "..."                              # dán từ ô 3
H = {"X-API-Key": KEY}

# 1) Đăng ký giọng mẫu: ref audio + transcript ĐÚNG của ref.
#    QUAN TRỌNG: transcript PHẢI khớp đúng lời trong ref.wav. Để TRỐNG "" tuy
#    KHÔNG nổ VRAM nhưng CHẤT LƯỢNG KÉM: model canh sai -> RÒ nguyên văn lời
#    trong ref vào bài đọc + LẶP/chèn chữ ở đầu đoạn. Không có transcript thì
#    tự phiên âm bằng faster-whisper (khớp tiếng, sạch):
#        # pip install faster-whisper
#        from faster_whisper import WhisperModel
#        segs, _ = WhisperModel("small", device="cuda", compute_type="float16"
#                     ).transcribe("ref.wav", language="vi", beam_size=5)
#        ref_text = " ".join(s.text.strip() for s in segs).strip()
ref_text = "transcript ĐÚNG của file ref.wav"   # hoặc ref_text từ whisper ở trên
audio, sr = sf.read("ref.wav")
buf = io.BytesIO(); sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
vid = requests.post(f"{URL}/voice", json={
    "name": "toi",
    "text": ref_text,
    "wav_b64": base64.b64encode(buf.getvalue()).decode(),
}, timeout=180, headers=H).json()["voice_id"]

# 2) Tạo giọng: gửi CẢ đoạn (tới 8000 ký tự), server tự chia -> giọng mượt.
r = requests.post(f"{URL}/tts", json={
    "text": "Câu một... Câu hai... (đoạn dài tuỳ ý, để engine tự chia).",
    "voice_id": vid, "lang": "Vietnamese", "steps": 32,
}, timeout=900, headers=H)
open("out.wav", "wb").write(r.content)
print("Xong -> out.wav  |  synth", r.headers.get("X-Synth-Seconds"), "s")
```

**Bài dài nhiều đoạn?** Dùng `/tts_batch` với `items=[{"i":0,"text":...}, ...]`
để gộp một request (đỡ trễ mạng). Mỗi item vẫn nên là cả đoạn, không cắt vụn.
